In [ ]:
!git clone https://github.com/mnielsen/neural-networks-and-deep-learning.git

In [ ]:
!ls neural-networks-and-deep-learning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Back up the classic neural network script
!cp network.py /content/drive/MyDrive/network.py

# 2. Back up the advanced neural network script (Cross-Entropy version)
# Note: If network2.py doesn't exist yet, this specific line will show a file warning.
# We will create network2.py next in the upcoming workflow step!
!cp network2.py /content/drive/MyDrive/network2.py

print("SUCCESS: Both network architectures have been permanently saved to your Google Drive root directory!")

In [ ]:
import pickle
import gzip
import numpy as np

# Defines a Python 3 compatible function to load data from the mnist.pkl.gz file
def load_data_wrapper_py3():
    # FIXED PATH: Points directly to the cloned repository directory
    with gzip.open('neural-networks-and-deep-learning/data/mnist.pkl.gz', 'rb') as f:
        # Using 'latin1' encoding to avoid byte-decoding errors in Python 3
        training_data, validation_data, test_data = pickle.load(f, encoding='latin1')

    # Format the training data for the neural network architecture
    def train_inputs(d): return [np.reshape(x, (784, 1)) for x in d[0]]
    def train_results(j):
        e = np.zeros((10, 1))
        e[j] = 1.0
        return e

    training_inputs = train_inputs(training_data)
    training_results = [train_results(y) for y in training_data[1]]
    training_d = list(zip(training_inputs, training_results))

    validation_inputs = [np.reshape(x, (784, 1)) for x in validation_data[0]]
    validation_d = list(zip(validation_inputs, validation_data[1]))

    test_inputs = [np.reshape(x, (784, 1)) for x in test_data[0]]
    test_d = list(zip(test_inputs, test_data[1]))

    return training_d, validation_d, test_d

# Load the dataset using the wrapper function
training_data, validation_data, test_data = load_data_wrapper_py3()
print("SUCCESS: MNIST data loaded into memory for Python 3!")

In [ ]:
import sys
import importlib

# 1. Define the complete Python 3 optimized script for network.py
python3_network_code = """
import random
import numpy as np

class Network(object):

    def __init__(self, sizes):
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x) for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a)+b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta, test_data=None):
        training_data = list(training_data)
        n = len(training_data)

        if test_data:
            test_data = list(test_data)
            n_test = len(test_data)

        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [
                training_data[k:k+mini_batch_size]
                for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            if test_data:
                print("Epoch {0}: {1} / {2}".format(j, self.evaluate(test_data), n_test))
            else:
                print("Epoch {0} complete".format(j))

    def update_mini_batch(self, mini_batch, eta):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb+dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw+dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w-(eta/len(mini_batch))*nw
                        for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b-(eta/len(mini_batch))*nb
                       for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        activation = x
        activations = [x]
        zs = []
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation)+b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
        return (nabla_b, nabla_w)

    def evaluate(self, test_data):
        test_results = [(np.argmax(self.feedforward(x)), y) for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def cost_derivative(self, output_activations, y):
        return (output_activations-y)

def sigmoid(z):
    return 1.0/(1.0+np.exp(-z))

def sigmoid_prime(z):
    return sigmoid(z)*(1.0-sigmoid(z))
"""

# 2. Overwrite the file on disk with our clean, Python 3-compatible code
with open('network.py', 'w') as file:
    file.write(python3_network_code)

print("SUCCESS: network.py rewritten and fully updated for Python 3 runtime environments!")

# 3. Force a clean modules cache flush and memory refresh
import network
importlib.reload(network)

# 4. Instantiate the neural network model object
net = network.Network([784, 30, 10])

# 5. Execute model training routine using SGD
print("Starting initial network model training optimization workflow...")
net.SGD(training_data, 30, 10, 3.0, test_data=test_data)

In [ ]:
# Inicializamos la red de 100 neuronas ocultas
net = network.Network([784, 100, 10])

# Bajamos el learning rate a 0.5 (en lugar de 3.0) para que los pasos sean finos
net.SGD(training_data, 30, 10, 0.5, test_data=test_data)

In [ ]:
import sys
import importlib
import base64
import io
import numpy as np
import matplotlib.pyplot as plt
from google.colab import output
from IPython.display import HTML, display

# Ensure the classic network module is loaded correctly
import network
importlib.reload(network)

# =====================================================================
# 1. LIVE HANDWRITTEN DIGIT INFERENCE USING THE OPTIMIZED MODEL
# =====================================================================
# Interactive HTML/JavaScript canvas for drawing inside the Colab cell
canvas_html = """
<canvas width="280" height="280" style="border:5px solid #4CAF50; background-color:black; cursor:crosshair;"></canvas>
<br><br>
<button id="predict_btn" style="padding:12px 24px; background-color:#4CAF50; color:white; font-size:16px; border:none; border-radius:4px; cursor:pointer; font-weight:bold;">Predict Digit!</button>
<script>
var canvas = document.querySelector('canvas');
var ctx = canvas.getContext('2d');
ctx.strokeStyle = 'white';
ctx.lineWidth = 24;  // Emulates MNIST line thickness
ctx.lineCap = 'round';
ctx.lineJoin = 'round';

var drawing = false;

canvas.addEventListener('mousedown', function(e) { drawing = true; ctx.beginPath(); ctx.moveTo(e.offsetX, e.offsetY); });
canvas.addEventListener('mousemove', function(e) { if (drawing) { ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke(); } });
canvas.addEventListener('mouseup', function() { drawing = false; });
canvas.addEventListener('mouseleave', function() { drawing = false; });

var button = document.querySelector('#predict_btn');
var p = new Promise(function(resolve, reject) {
    button.addEventListener('click', function() {
        resolve(canvas.toDataURL('image/png'));
    });
});
</script>
"""

print("Draw a digit (0-9) inside the black box, then click the green button:\n")
display(HTML(canvas_html))

# Capture drawing data via JavaScript promise injection
img_data = output.eval_js('p')
metadata, base64_data = img_data.split(',')
img_bytes = base64.b64decode(base64_data)

# =====================================================================
# 2. ADVANCED IMAGE PROCESSING (PADDING & BOUNDING BOX CENTERING)
# =====================================================================
from PIL import Image

# Convert raw canvas stream to grayscale image matrix
image = Image.open(io.BytesIO(img_bytes)).convert('L')

# Crop the canvas down to the bounding box of the actual drawing
bbox = image.getbbox()
if bbox:
    cropped_image = image.crop(bbox)

    # Calculate a proportional 25% padding border to mirror MNIST specifications
    max_dim = max(cropped_image.size)
    padding = int(max_dim * 0.25)

    # Generate a pristine black square background and paste the cropped digit dead center
    square_size = max_dim + 2 * padding
    padded_image = Image.new('L', (square_size, square_size), 0)
    padded_image.paste(cropped_image, ((square_size - cropped_image.size[0]) // 2,
                                       (square_size - cropped_image.size[1]) // 2))

    # Downsample safely to standard 28x28 pixel resolution using Lanczos filter
    image = padded_image.resize((28, 28), Image.Resampling.LANCZOS)
else:
    image = image.resize((28, 28), Image.Resampling.LANCZOS)

# Normalize pixel values mapping integer shades [0, 255] to floats [0.0, 1.0]
img_array = np.array(image) / 255.0

# Flatten 28x28 2D matrix into a 784x1 column vector structure for feeding the network
input_vector = np.reshape(img_array, (784, 1))

# =====================================================================
# 3. FORWARD PASS EXECUTION & PREDICTION OUTPUT
# =====================================================================
# Run inference using the first trained network ('net')
predictions = net.feedforward(input_vector)
predicted_digit = np.argmax(predictions)

# Output prediction narrative to evaluation terminal
print("\n" + "="*50)
print(f"   Neural Network Prediction: {predicted_digit}   ")
print("="*50 + "\n")

# Display low-resolution normalized viewport matrix
plt.figure(figsize=(3, 3))
plt.imshow(img_array, cmap='gray')
plt.title("Processed Input Viewport (28x28)", fontsize=10)
plt.axis('off')
plt.show()